# IDHM (Atlas / Base dos Dados)

Prepares the municipal Human Development Index used by the IVS/IDH book.
Variable definitions for the published book are in `dictionary.csv`.


## Imports

In [ ]:
import re
import numpy as np
import pandas as pd

from pathlib import Path
from typing import Optional
from unidecode import unidecode


## Configuração Inicial

In [ ]:
def resolve_repo_root() -> Path:
    """Locate the project root that contains the local data directory."""
    start = Path.cwd().resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "data").is_dir() and (candidate / "books").is_dir():
            return candidate
    return start


BASE_DIR = resolve_repo_root() / "data"

RAW_IDHM_DIR = BASE_DIR / "raw" / "idhm"
TRUSTED_IDHM_DIR = BASE_DIR / "trusted" / "idhm"
TRUSTED_IBGE_DIR = BASE_DIR / "trusted" / "ibge"
ANALYTICS_DIR = BASE_DIR / "analytics"

for path in [RAW_IDHM_DIR, TRUSTED_IDHM_DIR, ANALYTICS_DIR]:
    path.mkdir(parents=True, exist_ok=True)


## Leitura de Arquivos

In [ ]:
path_municipios_bd = RAW_IDHM_DIR / "br_bd_diretorios_brasil_municipio.csv.gz"
path_idhm_bd = RAW_IDHM_DIR / "mundo_onu_adh_municipio.csv.gz"

df_municipios_bd = pd.read_csv(path_municipios_bd, dtype="string", compression="gzip")
df_idhm_raw = pd.read_csv(path_idhm_bd, dtype="string", compression="gzip")


In [ ]:
print(df_municipios_bd.shape)
print(df_idhm_raw.shape)

display(df_municipios_bd.head())
display(df_idhm_raw.head())


In [ ]:
df_municipios_bd.columns.tolist()


In [ ]:
df_idhm_raw.columns.tolist()


## Normalizar nomes das colunas

In [ ]:
def normalizar_nome_coluna(col: str) -> str:
    col = str(col).strip()
    col = unidecode(col)
    col = col.lower()
    col = re.sub(r"[^a-z0-9]+", "_", col)
    col = re.sub(r"_+", "_", col)
    col = col.strip("_")
    return col


def normalizar_colunas(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [normalizar_nome_coluna(col) for col in df.columns]
    return df


In [ ]:
df_municipios_bd = normalizar_colunas(df_municipios_bd)
df_idhm_raw = normalizar_colunas(df_idhm_raw)

print(df_municipios_bd.columns.tolist())
print(df_idhm_raw.columns.tolist())


## Preparar a dimensão de municípios da Base dos Dados

In [ ]:
def preparar_dim_municipios_bd(df_municipios_bd: pd.DataFrame) -> pd.DataFrame:
    """
    Prepara dimensão municipal da Base dos Dados.

    Espera uma coluna id_municipio.
    Retorna uma dimensão com codigo_ibge e informações territoriais.
    """
    df = df_municipios_bd.copy()
    df = normalizar_colunas(df)

    if "id_municipio" not in df.columns:
        raise ValueError(f"Coluna id_municipio não encontrada. Colunas disponíveis: {df.columns.tolist()}")

    df["codigo_ibge"] = (
        df["id_municipio"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    # Renomeia colunas comuns da Base dos Dados, se existirem
    rename_map = {}

    if "nome" in df.columns:
        rename_map["nome"] = "municipio"

    if "sigla_uf" in df.columns:
        rename_map["sigla_uf"] = "uf"

    if "nome_uf" in df.columns:
        rename_map["nome_uf"] = "nome_uf"

    if "regiao" in df.columns:
        rename_map["regiao"] = "regiao"

    if "nome_regiao_imediata" in df.columns:
        rename_map["nome_regiao_imediata"] = "regiao_imediata"

    if "nome_regiao_intermediaria" in df.columns:
        rename_map["nome_regiao_intermediaria"] = "regiao_intermediaria"

    df = df.rename(columns=rename_map)

    cols_keep = [
        "codigo_ibge",
        "id_municipio",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "regiao_imediata",
        "regiao_intermediaria",
        "id_uf",
        "id_regiao_imediata",
        "id_regiao_intermediaria",
        "id_municipio_6",
    ]

    cols_keep = [col for col in cols_keep if col in df.columns]

    df = (
        df[cols_keep]
        .drop_duplicates("codigo_ibge")
        .sort_values("codigo_ibge")
        .reset_index(drop=True)
    )

    return df


In [ ]:
dim_municipios_bd = preparar_dim_municipios_bd(df_municipios_bd)

dim_municipios_bd.head()


## Salvar

In [ ]:
# dim_municipios_bd.to_parquet(
#     TRUSTED_IDHM_DIR / "dim_municipios_base_dos_dados.parquet",
#     index=False
# )

# dim_municipios_bd.to_csv(
#     TRUSTED_IDHM_DIR / "dim_municipios_base_dos_dados.csv",
#     index=False,
#     encoding="utf-8"
# )


## Base IDHM

In [ ]:
def selecionar_coluna(
    df: pd.DataFrame,
    candidatos: list[str],
    obrigatoria: bool = True,
    nome_variavel: str = ""
) -> Optional[str]:
    cols = set(df.columns)
    candidatos_norm = [normalizar_nome_coluna(c) for c in candidatos]

    for col in candidatos_norm:
        if col in cols:
            return col

    if obrigatoria:
        raise ValueError(
            f"Não encontrei coluna para {nome_variavel}. "
            f"Candidatos testados: {candidatos_norm}. "
            f"Colunas disponíveis: {list(df.columns)}"
        )

    return None


def parse_decimal(s: pd.Series) -> pd.Series:
    """
    Parse decimal strings using comma or dot as decimal separator.
    """
    return (
        s.astype("string")
        .str.strip()
        .replace({
            "": pd.NA,
            "nan": pd.NA,
            "None": pd.NA,
            "-": pd.NA,
            "...": pd.NA,
        })
        .str.replace(",", ".", regex=False)
        .pipe(pd.to_numeric, errors="coerce")
    )


In [ ]:
def padronizar_idhm_base_dos_dados(df_idhm_raw: pd.DataFrame) -> pd.DataFrame:
    """
    Padroniza a base mundo_onu_adh_municipio baixada da Base dos Dados.

    Saída:
        codigo_ibge
        ano
        idhm
        idhm_educacao
        idhm_longevidade
        idhm_renda
    """
    df = normalizar_colunas(df_idhm_raw)

    col_id_municipio = selecionar_coluna(
        df,
        ["id_municipio", "codigo_ibge", "cod_ibge"],
        obrigatoria=True,
        nome_variavel="id_municipio"
    )

    col_ano = selecionar_coluna(
        df,
        ["ano"],
        obrigatoria=True,
        nome_variavel="ano"
    )

    col_idhm = selecionar_coluna(
        df,
        ["idhm", "idh_m"],
        obrigatoria=True,
        nome_variavel="idhm"
    )

    col_idhm_educacao = selecionar_coluna(
        df,
        ["idhm_educacao", "idhm_e", "idh_m_educacao"],
        obrigatoria=False,
        nome_variavel="idhm_educacao"
    )

    col_idhm_longevidade = selecionar_coluna(
        df,
        ["idhm_longevidade", "idhm_l", "idh_m_longevidade"],
        obrigatoria=False,
        nome_variavel="idhm_longevidade"
    )

    col_idhm_renda = selecionar_coluna(
        df,
        ["idhm_renda", "idhm_r", "idh_m_renda"],
        obrigatoria=False,
        nome_variavel="idhm_renda"
    )

    out = pd.DataFrame()

    out["codigo_ibge"] = (
        df[col_id_municipio]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    out["ano"] = pd.to_numeric(
        df[col_ano],
        errors="coerce"
    ).astype("Int64")

    out["idhm"] = parse_decimal(df[col_idhm])

    out["idhm_educacao"] = (
        parse_decimal(df[col_idhm_educacao])
        if col_idhm_educacao is not None
        else np.nan
    )

    out["idhm_longevidade"] = (
        parse_decimal(df[col_idhm_longevidade])
        if col_idhm_longevidade is not None
        else np.nan
    )

    out["idhm_renda"] = (
        parse_decimal(df[col_idhm_renda])
        if col_idhm_renda is not None
        else np.nan
    )

    out = (
        out
        .dropna(subset=["codigo_ibge", "ano", "idhm"])
        .drop_duplicates(["codigo_ibge", "ano"])
        .sort_values(["codigo_ibge", "ano"])
        .reset_index(drop=True)
    )

    return out


In [ ]:
idhm_municipio = padronizar_idhm_base_dos_dados(df_idhm_raw)

idhm_municipio.head()


In [ ]:
(
    idhm_municipio
    .groupby("ano")
    .agg(
        qtd_municipios=("codigo_ibge", "nunique"),
        idhm_medio=("idhm", "mean"),
        idhm_min=("idhm", "min"),
        idhm_max=("idhm", "max"),
        idhm_educacao_medio=("idhm_educacao", "mean"),
        idhm_longevidade_medio=("idhm_longevidade", "mean"),
        idhm_renda_medio=("idhm_renda", "mean"),
    )
    .reset_index()
    .sort_values("ano")
)


## IDHM e Base dos Dados

In [ ]:
idhm_municipio_bd = (
    idhm_municipio
    .merge(
        dim_municipios_bd,
        on="codigo_ibge",
        how="left"
    )
)

idhm_municipio_bd.head()


In [ ]:
cols_ordem = [
    "codigo_ibge",
    "id_municipio",
    "ano",
    "municipio",
    "uf",
    "nome_uf",
    "regiao",
    "regiao_imediata",
    "regiao_intermediaria",
    "idhm",
    "idhm_educacao",
    "idhm_longevidade",
    "idhm_renda",
]

cols_ordem = [col for col in cols_ordem if col in idhm_municipio_bd.columns]

idhm_municipio_bd = idhm_municipio_bd[
    cols_ordem + [col for col in idhm_municipio_bd.columns if col not in cols_ordem]
]

idhm_municipio_bd.head()


In [ ]:
# idhm_municipio.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_tratado.parquet",
#     index=False
# )

# idhm_municipio_bd.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_diretorio_bd.parquet",
#     index=False
# )

# idhm_municipio_bd.to_csv(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_diretorio_bd.csv",
#     index=False,
#     encoding="utf-8"
# )


## Cruzar com dim_municipio do IBGE
Há 4 ou 5 referências nos municípios e 3 referências de IDHM. Precisa corrigir.

In [ ]:
def carregar_dim_municipio_ibge(
    path_dim: Path = TRUSTED_IBGE_DIR / "dim_municipio.parquet"
) -> pd.DataFrame:
    """
    Carrega a dimensão oficial de municípios do IBGE usada no projeto.
    """
    dim = pd.read_parquet(path_dim).copy()

    dim["codigo_ibge"] = (
        dim["codigo_ibge"]
        .astype("string")
        .str.replace(r"\D", "", regex=True)
        .str.zfill(7)
    )

    cols = [
        "codigo_ibge",
        "municipio",
        "uf",
        "nome_uf",
        "regiao",
        "sigla_regiao",
        "regiao_imediata",
        "regiao_intermediaria",
        "codigo_uf",
        "codigo_regiao",
        "codigo_regiao_imediata",
        "codigo_regiao_intermediaria",
    ]

    cols = [col for col in cols if col in dim.columns]

    return dim[cols].drop_duplicates("codigo_ibge")


In [ ]:
dim_municipio_ibge = carregar_dim_municipio_ibge(
    TRUSTED_IBGE_DIR / "dim_municipio.parquet"
)

idhm_municipio_ibge = (
    idhm_municipio
    .merge(
        dim_municipio_ibge,
        on="codigo_ibge",
        how="left"
    )
)

idhm_municipio_ibge.head()


In [ ]:
# idhm_municipio_ibge.to_parquet(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_dim_ibge.parquet",
#     index=False
# )

# idhm_municipio_ibge.to_csv(
#     TRUSTED_IDHM_DIR / "idhm_municipio_com_dim_ibge.csv",
#     index=False,
#     encoding="utf-8"
# )
